In [1]:
import argparse
import json
import os
import re
import time
from typing import Dict, List, Optional, Tuple

import numpy as np
from openai import OpenAI, BadRequestError, APIConnectionError, APITimeoutError, RateLimitError

from train_phase1 import (Curriculum, CURRICULUMS, TASK_NAMES, TASK_SPECS, N_TASKS, SEEDS, 
                          EVAL_FREQ, N_EVAL_EPISODES_PER_ZONE, SNAPSHOT_EVERY, train_one_run,
                          _overall_success, _slug)
TOTAL_TIMESTEPS=20_000

class LLMClient:
    def __init__(self, base_url: str, model: str = "Qwen/Qwen2.5-14B-Instruct", api_key: str = "not-needed",
                 temperature: float = 0.2, max_tokens: int = 700, max_retries: int = 3, timeout: float = 120.0):
        self.client = OpenAI(base_url=base_url, api_key=api_key, timeout=timeout)
        self.model = model
        self.temperature = temperature
        self.max_tokens = max_tokens
        self.max_retries = max_retries

    def generate_json(self, prompt: str) -> str:
        kwargs = dict(model=self.model, messages=[{"role": "user", "content": prompt}],
                      temperature=self.temperature, max_tokens=self.max_tokens)
        last_exc: Optional[Exception] = None
        for attempt in range(self.max_retries):
            try:
                try:
                    response = self.client.chat.completions.create(response_format={"type": "json_object"}, **kwargs)
                except BadRequestError:
                    response = self.client.chat.completions.create(**kwargs)
                return response.choices[0].message.content or ""
            except (APIConnectionError, APITimeoutError, RateLimitError) as exc:
                last_exc = exc
                time.sleep(2.0 * (attempt + 1))
        raise last_exc


LADDER_DESCRIPTION = (
    "A robot arm is trained on 5 manipulation tasks: reach, push, slide, pickandplace, stack. "
    "reach/push/slide use a fixed, non-grasping gripper. pickandplace and stack require grasping; "
    "stack needs essentially the same grasp-and-place competence as pickandplace, and both are only "
    "learnable once basic reaching/pushing competence exists. Each task is split into 3 distance "
    "levels (near, medium, far) = how far the target is from the centre of that task's target region. "
    "Every (task, distance_level) pair is a 'bin'. Success = the object/gripper ends within a small "
    "threshold of the target at the end of an episode."
)


def sample_on_shell(spec, frac: float, rng: np.random.Generator) -> np.ndarray:
    low, high = spec.goal_low, spec.goal_high
    free = (high - low) > 1e-9
    center = (low + high) / 2.0
    half_diag = np.linalg.norm(((high - low) / 2.0)[free])
    g = rng.normal(size=int(free.sum()))
    direction = np.zeros(len(low))
    direction[free] = g / max(np.linalg.norm(g), 1e-9)
    return np.clip(center + direction * frac * half_diag, low, high).astype(np.float32)


class SingleAgentLLMCurriculum(Curriculum):
    DIFFICULTY_LEVELS = ["near", "medium", "far"]
    DIFFICULTY_TO_FRACTION = [0.20, 0.55, 0.80]
    WINDOW = 20
    MAX_LLM_ERRORS = 20
    def __init__(self, llm_client: LLMClient, revision_freq: int = 10, band: float = 0.15,
                 nb_obj_prompt: int = 10, log_path: Optional[str] = None):
        self.llm_client = llm_client
        self.revision_freq = revision_freq
        self.band = band
        self.nb_obj_prompt = nb_obj_prompt
        self.log_path = log_path
        self.bins = [(t, l) for t in TASK_NAMES for l in self.DIFFICULTY_LEVELS]
        self.bin_index = {b: i for i, b in enumerate(self.bins)}
        self.history: Dict[int, List[bool]] = {i: [] for i in range(len(self.bins))}
        self.plan: List[int] = []
        self.pos = 0
        self.last_bin: Optional[int] = None
        self.episode_count = 0
        self.n_llm_calls = 0
        self.n_revisions = 0
        self.n_fallbacks = 0
        self.n_invalid_items = 0
        self.n_llm_errors = 0

    def _bin_stats(self) -> Dict[str, Dict]:
        w = self.WINDOW
        stats = {}
        for i, (task, level) in enumerate(self.bins):
            h = self.history[i]
            recent, previous = h[-w:], h[-2 * w:-w]
            entry = {"episodes": len(h),
                     "success_recent": round(float(np.mean(recent)), 2) if recent else None}
            if len(previous) >= 5:
                entry["success_previous"] = round(float(np.mean(previous)), 2)
            stats[f"{task}/{level}"] = entry
        return stats

    def build_prompt(self) -> str:
        return (
            f"{LADDER_DESCRIPTION}\n\n"
            "You manage the training curriculum. Rolling success rate per bin over the last "
            f"{self.WINDOW} training episodes in that bin ('success_previous' = the {self.WINDOW} before; "
            "null = not attempted yet):\n"
            f"{json.dumps(self._bin_stats(), indent=2)}\n\n"
            f"Propose a list of {self.nb_obj_prompt} bins for the next episodes, to make the fastest overall "
            "progress across all 5 tasks. A bin may appear several times to get more episodes. "
            'Respond with ONLY a JSON object: {"bins": [{"task": "<reach|push|slide|pickandplace|stack>", '
            '"distance_level": "<near|medium|far>", "reason": "<one short sentence>"}, ...]}'
        )

    def _parse_llm_choice(self, raw: str) -> List[int]:
        text = re.sub(r"^```(?:json)?\s*|\s*```$", "", (raw or "").strip(), flags=re.M).strip()
        obj = None
        for candidate in (text, *re.findall(r"\{.*\}|\[.*\]", text, flags=re.S)):
            try:
                obj = json.loads(candidate)
                break
            except (json.JSONDecodeError, TypeError):
                continue
        if isinstance(obj, dict):
            obj = obj.get("bins")
        if not isinstance(obj, list):
            return []
        idxs: List[int] = []
        for item in obj:
            pair = None
            if isinstance(item, dict):
                pair = (str(item.get("task", "")).strip().lower(), str(item.get("distance_level", "")).strip().lower())
            if pair in self.bin_index:
                idxs.append(self.bin_index[pair])
            else:
                self.n_invalid_items += 1
        return idxs

    def _call_llm(self, prompt: str) -> str:
        self.n_llm_calls += 1
        try:
            return self.llm_client.generate_json(prompt)
        except (APIConnectionError, APITimeoutError, RateLimitError):
            self.n_llm_errors += 1
            if self.n_llm_errors >= self.MAX_LLM_ERRORS:
                raise RuntimeError(f"{self.n_llm_errors} LLM calls failed: is the vLLM server still up?")
            return ""

    def _append_log(self, prompt: str, raw: str, idxs: List[int], fallback: bool) -> None:
        if not self.log_path:
            return
        record = {"episode": self.episode_count, "revision": self.n_revisions, "prompt": prompt, "raw": raw,
                  "plan": [f"{self.bins[i][0]}/{self.bins[i][1]}" for i in self.plan],
                  "parsed": [f"{self.bins[i][0]}/{self.bins[i][1]}" for i in idxs], "fallback": fallback,
                  "n_llm_calls": self.n_llm_calls, "n_fallbacks": self.n_fallbacks,
                  "n_invalid_items": self.n_invalid_items}
        with open(self.log_path, "a") as f:
            f.write(json.dumps(record) + "\n")

    def _revise(self) -> None:
        prompt = self.build_prompt()
        raw = self._call_llm(prompt)
        idxs = self._parse_llm_choice(raw)
        fallback = not idxs
        if idxs:
            self.plan = idxs[: self.nb_obj_prompt]
        else:
            self.n_fallbacks += 1
            if not self.plan:
                self.plan = list(range(min(self.nb_obj_prompt, len(self.bins))))
        self.n_revisions += 1
        self.pos = 0
        self._append_log(prompt, raw, idxs, fallback)
        if self.n_revisions <= 3 or self.n_revisions % 50 == 0:
            names = [f"{self.bins[i][0]}/{self.bins[i][1]}" for i in self.plan]
            print(f"[llm] revision {self.n_revisions} (episode {self.episode_count}, fallback={fallback}): {names}",
                  flush=True)

    def propose(self, rng: np.random.Generator) -> Tuple[int, np.ndarray]:
        if not self.plan or self.pos >= self.revision_freq:
            self._revise()
        bin_idx = self.plan[self.pos % len(self.plan)]
        self.pos += 1
        self.last_bin = bin_idx
        name, level = self.bins[bin_idx]
        spec = TASK_SPECS[name]
        frac_c = self.DIFFICULTY_TO_FRACTION[self.DIFFICULTY_LEVELS.index(level)]
        frac = rng.uniform(max(frac_c - self.band, 0.0), min(frac_c + self.band, 1.0))
        return spec.task_id, sample_on_shell(spec, frac, rng)

    def notify_result(self, task_id: int, success: bool) -> None:
        if self.last_bin is not None:
            self.history[self.last_bin].append(bool(success))
        self.episode_count += 1

    def snapshot(self) -> Dict:
        return {"plan": [f"{self.bins[i][0]}/{self.bins[i][1]}" for i in self.plan],
                "n_llm_calls": self.n_llm_calls, "n_revisions": self.n_revisions,
                "n_fallbacks": self.n_fallbacks}

    def llm_stats(self) -> Dict:
        return {"n_llm_calls": self.n_llm_calls, "n_revisions": self.n_revisions,
                "n_fallbacks": self.n_fallbacks, "n_invalid_items": self.n_invalid_items,
                "n_llm_errors": self.n_llm_errors, "episodes": self.episode_count,
                "revision_freq": self.revision_freq}
        








In [2]:
import subprocess
import time
import urllib.request
import sys

import vllm
print(vllm.__version__)
def start_vllm(model: str, port: int, log_file: str = "vllm.log", timeout: int = 900):
    log = open(log_file, "w")
    env = os.environ.copy()
    env["LD_LIBRARY_PATH"] = os.path.join(sys.prefix, "lib") + ":" + env.get("LD_LIBRARY_PATH", "")
    env["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

    proc = subprocess.Popen(
        [sys.executable, "-m", "vllm.entrypoints.openai.api_server",
         "--model", model,
         "--port", str(port),
         "--gpu-memory-utilization", "0.8" ,"--max-model-len", "8192"],
        stdout=log, stderr=subprocess.STDOUT,
        env=env,
    )


    url = f"http://127.0.0.1:{port}/v1/models"
    t0 = time.time()
    while time.time() - t0 < timeout:
        if proc.poll() is not None:          
            raise RuntimeError(f"vLLM s'est arrêté, regarde {log_file}")
        try:
            urllib.request.urlopen(url, timeout=2)
            print(f"serveur prêt sur le port {port} ({time.time() - t0:.0f}s)")
            return proc
        except Exception:
            time.sleep(5)                     

    proc.terminate()
    raise TimeoutError(f"vLLM pas prêt après {timeout}s, regarde {log_file}")

0.31.0


In [ ]:
CURRICULUMS["single-agent LLM"] = SingleAgentLLMCurriculum
STRATEGIES = list(CURRICULUMS.keys())

NAME = "single-agent LLM"
MODEL = "Qwen/Qwen2.5-14B-Instruct"
PORT = 8000
llm_url = f"http://127.0.0.1:{PORT}/v1"

server = start_vllm(MODEL, port=PORT, log_file="vllm.log")   # un seul serveur
try:
    llm = LLMClient(llm_url, MODEL)
    print(f"checking LLM server at {llm_url} ...", flush=True)
    test = llm.generate_json('Reply with the JSON object {"ok": true}')
    print(f"LLM server OK, answer: {test[:80]!r}", flush=True)

    for seed in SEEDS:
        results_path = os.path.join(".", "results", f"results_llm_single_seed{seed}.json")
        stats_path = os.path.join(".", "results", f"llm_stats_single_seed{seed}.json")
        goal_prefix = os.path.join(".", "results", "goals", f"{_slug(NAME)}_seed{seed}")
        log_path = os.path.join(".", "results", "goals", f"{_slug(NAME)}_seed{seed}_llm_log.jsonl")
        os.makedirs(os.path.dirname(goal_prefix), exist_ok=True)

        if os.path.exists(results_path):
            with open(results_path) as f:
                results = json.load(f)
        else:
            results = {}

        if NAME in results:
            print(f"skip {NAME!r} seed={seed} (already done)")
            print(f"success rate (mean over tasks): {_overall_success(results[NAME][-1]):.2f}")
            continue

        created: List[SingleAgentLLMCurriculum] = []

        def factory():
            curriculum = SingleAgentLLMCurriculum(llm, revision_freq=10, log_path=log_path)
            created.append(curriculum)
            return curriculum

        t0 = time.time()
        print(f"strategy={NAME}  seed={seed}  total_timesteps={TOTAL_TIMESTEPS}", flush=True)
        history = train_one_run(factory, seed, total_timesteps=TOTAL_TIMESTEPS, eval_freq=EVAL_FREQ,
                                n_episodes_per_zone=N_EVAL_EPISODES_PER_ZONE, verbose=0,
                                goal_log_path=goal_prefix, snapshot_every=SNAPSHOT_EVERY)

        results[NAME] = history
        with open(results_path, "w") as f:
            json.dump(results, f, indent=2)
        with open(stats_path, "w") as f:
            json.dump(created[0].llm_stats(), f, indent=2)
        print(f"success rate (mean over tasks): {_overall_success(history[-1]):.2f}")
        print(f"seed {seed} done in {time.time() - t0:.0f}s -> {results_path}")
finally:
    server.terminate()     
    server.wait()

serveur prêt sur le port 8000 (420s)
checking LLM server at http://127.0.0.1:8000/v1 ...
LLM server OK, answer: '{\n  "ok": true\n}'
skip 'single-agent LLM' seed=10 (already done)
success rate (mean over tasks): 0.23
skip 'single-agent LLM' seed=11 (already done)
success rate (mean over tasks): 0.22
strategy=single-agent LLM  seed=12  total_timesteps=20000


pybullet build time: Jan 29 2025 23:17:20


[llm] revision 1 (episode 0, fallback=False): ['reach/near', 'push/near', 'slide/near', 'reach/medium', 'push/medium', 'slide/medium', 'pickandplace/near', 'stack/near', 'reach/far', 'push/far']
[llm] revision 2 (episode 10, fallback=False): ['reach/near', 'reach/medium', 'reach/far', 'push/near', 'pickandplace/near', 'reach/medium', 'stack/near', 'push/medium', 'slide/near', 'reach/far']
[llm] revision 3 (episode 20, fallback=False): ['reach/near', 'reach/medium', 'reach/far', 'push/near', 'push/medium', 'push/far', 'slide/near', 'pickandplace/near', 'stack/near', 'reach/medium']


In [19]:
import json
# load goals HC2 seed 10:
hc2_path="results/goals/hand_crafted_2__domain_informed_seed10_snapshots.json"
with open(hc2_path) as f:
    hc2_goals=json.load(f)

# load goals single LLM seed 10:
llm_path="results/goals/single_agent_LLM_seed10_snapshots.json"
with open(llm_path) as f:
    llm_goals=json.load(f)

print([hc2_goals[i]['rolling_success']for i in range(10)])
print(llm_goals[0]['plan'])

[{'reach': 1.0, 'push': 0.0, 'slide': 0.0, 'pickandplace': 0.0, 'stack': 0.0}, {'reach': 1.0, 'push': 0.05, 'slide': 0.0, 'pickandplace': 0.0, 'stack': 0.0}, {'reach': 1.0, 'push': 0.0, 'slide': 0.0, 'pickandplace': 0.0, 'stack': 0.0}, {'reach': 1.0, 'push': 0.25, 'slide': 0.0, 'pickandplace': 0.0, 'stack': 0.0}, {'reach': 1.0, 'push': 0.05, 'slide': 0.0, 'pickandplace': 0.0, 'stack': 0.0}, {'reach': 1.0, 'push': 0.25, 'slide': 0.0, 'pickandplace': 0.0, 'stack': 0.0}, {'reach': 1.0, 'push': 0.15, 'slide': 0.05, 'pickandplace': 0.0, 'stack': 0.0}, {'reach': 1.0, 'push': 0.4, 'slide': 0.05, 'pickandplace': 0.0, 'stack': 0.0}, {'reach': 1.0, 'push': 0.3, 'slide': 0.05, 'pickandplace': 0.0, 'stack': 0.0}, {'reach': 1.0, 'push': 0.55, 'slide': 0.1, 'pickandplace': 0.0, 'stack': 0.0}]
['reach/far', 'reach/medium', 'push/near', 'push/medium', 'stack/near', 'stack/medium', 'pickandplace/near', 'slide/near', 'push/far', 'stack/far']
